# Setup
データ取り込みを行う。 DuckDB 想定。

In [ ]:
import os
import dotenv

dotenv.load_dotenv()
ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

# Extract

In [ ]:
from pathlib import Path
from assistant_agent.loaders import ObsidianReader

# Vault から読み込み
vault_path = Path("../../docs/dataset_obsidian/")
loader = ObsidianReader(vault_path)
docs = loader.load_data()

print(f"{len(docs)} 件のノートを読み込みました")

# Load

In [ ]:
from pathlib import Path

# 保存先ディレクトリを作成
vault_db_path = Path("../../tests/data/vault_db").resolve()
vault_db_path.mkdir(exist_ok=True)

In [ ]:
from sqlalchemy import create_engine, text
from assistant_agent.entities.duckdb import VaultBase, ObsidianEntity
from assistant_agent.entities.base import VaultUtils

# DB へ取り込み
path = vault_db_path / "entity.duckdb"
sa_engine = create_engine(f"duckdb:///{path}")

with sa_engine.connect() as sess:
    sess.execute(text("create schema if not exists assets;"))
    sess.commit()

VaultBase.metadata.create_all(sa_engine)
VaultUtils.sync(docs, sa_engine, ObsidianEntity)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
from pprint import pprint
from sqlalchemy import text

with sa_engine.connect() as sess:
    res = sess.execute(text("select * from entity.assets.obsidian_raw limit 3"))
    pprint(res.all())

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

from assistant_agent.entities.duckdb import ObsidianEntity
from assistant_agent.services import VaultObsidianRetriever
from assistant_agent.utils.store_context import DuckDBStoreContext

# レトリーバーを作成
store_ctx = DuckDBStoreContext(vault_db_path)
sa_engine = store_ctx.get_engine()
obsidian_retriever = VaultObsidianRetriever(
    "obsidian_docstore",
    "obsidian_vectors",
    store_context=store_ctx,
    transformations=[
        SentenceSplitter(
            chunk_size=1024,
            chunk_overlap=200,
            paragraph_separator="\n\n",
        ),
    ],
    embed_model=GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=ENV_GEMINI_API_KEY,
    ),
    embed_dim=3072,
    vault_entity=ObsidianEntity,
)

In [ ]:
obsidian_retriever.sync_chunks()

In [ ]:
# list(store_ctx._conn.execute("SHOW DATABASES").fetchall())
# list(store_ctx._conn.execute("SHOW TABLES").fetchall())
list(store_ctx._conn.execute("SELECT * FROM obsidian_docstore LIMIT 3").fetchall())
# list(store_ctx._conn.execute("SELECT * FROM obsidian_vectors LIMIT 3").fetchall())

# Retrieval

In [ ]:
from llama_index.core.vector_stores.types import (
    VectorStoreQuery, VectorStoreQueryMode,
    MetadataFilters, MetadataFilter, FilterOperator
)
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding


ember = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=os.getenv("ENV_GEMINI_API_KEY"))
query_res = obsidian_retriever._vector_store.query(
    VectorStoreQuery(
        query_embedding=ember.get_query_embedding("プロンプトエンジニアリング"),
        similarity_top_k=5,
        mode=VectorStoreQueryMode.MMR,
        mmr_threshold=0.5,
        filters=MetadataFilters(
            filters=[
                MetadataFilter(key="file_path", value="03_Structure/", operator=FilterOperator.TEXT_MATCH)
            ]
        )
    )
)
assert query_res.nodes is not None
for item in query_res.nodes:
    print(item.text)  # pyright: ignore[reportAttributeAccessIssue]
    print("==================")